# Experiments 1,2
## Date: 18-11-2026

# 1.Data Collection, Storage and Retrieval from Files

In [1]:
import csv 
import json 
import os 
import pandas as pd 
  
os.makedirs("data", exist_ok=True) 
  
# ---------- Step 1: Data collection from three different sources ---------- 
# Source 1: CSV file exported from the in-store billing (POS) system 
pos_rows = [ 
    ["order_id", "customer", "product", "qty", "unit_price", "branch"], 
    [101, "Anitha", "Laptop", 1, 54000, "Chennai"], 
    [102, "Rahul", "Mouse", 3, 450, "Chennai"], 
    [103, "Priya", "Keyboard", 2, 1200, "Chennai"], 
    [104, "Karthik", "Monitor", 1, 9500, "Chennai"], 
] 
with open("data/pos_orders.csv", "w", newline="") as f: 
    csv.writer(f).writerows(pos_rows) 
  
# Source 2: JSON response from the online store (web API) 
web_orders = [ 
    { 
        "order_id": 201, 
        "customer": "Divya", 
        "product": "Laptop", 
        "qty": 2, 
        "unit_price": 52500, 
        "branch": "Online", 
    }, 
    { 
        "order_id": 202, 
        "customer": "Arjun", 
        "product": "Headset", 
        "qty": 1, 
        "unit_price": 2300, 
        "branch": "Online", 
    }, 
    { 
        "order_id": 203, 
        "customer": "Meena", 
        "product": "Mouse", 
        "qty": 5, 
        "unit_price": 430, 
        "branch": "Online", 
    }, 
] 
with open("data/web_orders.json", "w") as f: 
    json.dump(web_orders, f, indent=2) 
  
# Source 3: Plain-text log written by the mobile app (pipe separated) 
log_lines = [ 
    "301|Suresh|Monitor|2|9300|Mobile", 
    "302|Lakshmi|Keyboard|1|1150|Mobile", 
    "303|Vignesh|Laptop|1|55000|Mobile", 
] 
with open("data/mobile_orders.txt", "w") as f: 
    f.write("\n".join(log_lines)) 
  
print("Retail Order Data: Collection, Storage and Retrieval") 
print("=" * 55) 
  
# ---------- Step 2: Reading the data back from each file ---------- 
df_pos = pd.read_csv("data/pos_orders.csv") 
print("\n[1] CSV source  -> rows:", len(df_pos)) 
print(df_pos.to_string(index=False)) 
  
with open("data/web_orders.json") as f: 
    df_web = pd.DataFrame(json.load(f)) 
print("\n[2] JSON source -> rows:", len(df_web)) 
print(df_web.to_string(index=False)) 
  
cols = ["order_id", "customer", "product", "qty", "unit_price", "branch"] 
df_mob = pd.read_csv("data/mobile_orders.txt", sep="|", header=None, names=cols) 
print("\n[3] TXT source  -> rows:", len(df_mob)) 
print(df_mob.to_string(index=False)) 
  
# ---------- Step 3: Integration into a single dataset ---------- 
orders = pd.concat([df_pos, df_web, df_mob], ignore_index=True) 
orders["amount"] = orders["qty"] * orders["unit_price"] 
print("\n--- Integrated Dataset ---") 
print(orders.to_string(index=False)) 
print("\nShape        :", orders.shape) 
print("Data types   :") 
print(orders.dtypes.to_string()) 
  
# ---------- Step 4: Storage in different formats ---------- 
orders.to_csv("data/all_orders.csv", index=False) 
orders.to_json("data/all_orders.json", orient="records", indent=2) 
orders.to_excel("data/all_orders.xlsx", index=False) 
print("\nFiles stored :") 
for name in ["all_orders.csv", "all_orders.json", "all_orders.xlsx"]: 
    print(f"  data/{name:<16} {os.path.getsize('data/' + name):>6} bytes") 
  
# ---------- Step 5: Retrieval and a quick query ---------- 
back = pd.read_excel("data/all_orders.xlsx") 
print("\nRetrieved from Excel, rows:", len(back)) 
print("\nRevenue by branch:") 
print( 
    back.groupby("branch")["amount"] 
    .sum() 
    .sort_values(ascending=False) 
    .to_string() 
) 
print("\nOrders with amount > 10000:") 
print( 
    back.loc[ 
        back["amount"] > 10000, ["order_id", "customer", "product", "amount"] 
    ].to_string(index=False) 
) 

Retail Order Data: Collection, Storage and Retrieval

[1] CSV source  -> rows: 4
 order_id customer  product  qty  unit_price  branch
      101   Anitha   Laptop    1       54000 Chennai
      102    Rahul    Mouse    3         450 Chennai
      103    Priya Keyboard    2        1200 Chennai
      104  Karthik  Monitor    1        9500 Chennai

[2] JSON source -> rows: 3
 order_id customer product  qty  unit_price branch
      201    Divya  Laptop    2       52500 Online
      202    Arjun Headset    1        2300 Online
      203    Meena   Mouse    5         430 Online

[3] TXT source  -> rows: 3
 order_id customer  product  qty  unit_price branch
      301   Suresh  Monitor    2        9300 Mobile
      302  Lakshmi Keyboard    1        1150 Mobile
      303  Vignesh   Laptop    1       55000 Mobile

--- Integrated Dataset ---
 order_id customer  product  qty  unit_price  branch  amount
      101   Anitha   Laptop    1       54000 Chennai   54000
      102    Rahul    Mouse    3    

# 2.Working with Relational Databases and NoSQL Documents

In [2]:
import sqlite3 
import pandas as pd 
  
conn = sqlite3.connect(":memory:") 
cur = conn.cursor() 
  
# ---------- Step 1: Create the relational schema ---------- 
cur.executescript(""" 
CREATE TABLE doctors ( 
    doctor_id   INTEGER PRIMARY KEY, 
    name        TEXT NOT NULL, 
    department  TEXT NOT NULL 
); 
CREATE TABLE patients ( 
    patient_id  INTEGER PRIMARY KEY, 
    name        TEXT NOT NULL, 
    age         INTEGER, 
    gender      TEXT, 
    city        TEXT 
); 
CREATE TABLE visits ( 
    visit_id    INTEGER PRIMARY KEY, 
    patient_id  INTEGER REFERENCES patients(patient_id), 
    doctor_id   INTEGER REFERENCES doctors(doctor_id), 
    visit_date  TEXT, 
    fee         REAL 
); 
""") 
  
# ---------- Step 2: Insert records ---------- 
cur.executemany( 
    "INSERT INTO doctors VALUES (?,?,?)", 
    [ 
        (1, "Dr. Ramesh", "Cardiology"), 
        (2, "Dr. Kavitha", "Orthopaedics"), 
        (3, "Dr. Imran", "General Medicine"), 
    ], 
) 
cur.executemany( 
    "INSERT INTO patients VALUES (?,?,?,?,?)", 
    [ 
        (1, "Ajay", 45, "M", "Chennai"), 
        (2, "Bhavana", 32, "F", "Madurai"), 
        (3, "Charles", 67, "M", "Chennai"), 
        (4, "Deepa", 29, "F", "Coimbatore"), 
        (5, "Elango", 54, "M", "Madurai"), 
        (6, "Fathima", 38, "F", "Chennai"), 
    ], 
) 
cur.executemany( 
    "INSERT INTO visits VALUES (?,?,?,?,?)", 
    [ 
        (1, 1, 1, "2026-01-05", 800), 
        (2, 2, 2, "2026-01-06", 600), 
        (3, 3, 1, "2026-01-08", 800), 
        (4, 4, 3, "2026-01-09", 300), 
        (5, 5, 1, "2026-01-12", 800), 
        (6, 1, 3, "2026-01-15", 300), 
        (7, 6, 2, "2026-01-18", 600), 
        (8, 3, 3, "2026-01-20", 300), 
        (9, 2, 2, "2026-01-22", 600), 
        (10, 5, 3, "2026-01-25", 300), 
    ], 
) 
conn.commit() 
  
  
def run(title, sql): 
    print(f"\n--- {title} ---") 
    print("SQL:", " ".join(sql.split())) 
    print(pd.read_sql_query(sql, conn).to_string(index=False)) 
  
  
print("Hospital Patient Records: Working with a Relational Database") 
print("=" * 60) 
  
run( 
    "Q1. Patients older than 40", 
    """ 
    SELECT name, age, city FROM patients WHERE age > 40 ORDER BY age DESC""", 
) 
  
run( 
    "Q2. Visits with patient and doctor names (JOIN)", 
    """ 
    SELECT v.visit_id, p.name AS patient, d.name AS doctor, d.department, v.fee 
    FROM visits v 
    JOIN patients p ON v.patient_id = p.patient_id 
    JOIN doctors  d ON v.doctor_id  = d.doctor_id 
    ORDER BY v.visit_id""", 
) 
  
run( 
    "Q3. Revenue and visit count per department (GROUP BY)", 
    """ 
    SELECT d.department, COUNT(*) AS visits, SUM(v.fee) AS revenue 
    FROM visits v JOIN doctors d ON v.doctor_id = d.doctor_id 
    GROUP BY d.department ORDER BY revenue DESC""", 
) 
  
run( 
    "Q4. Patients with more than one visit (HAVING)", 
    """ 
    SELECT p.name, COUNT(*) AS visits 
    FROM visits v JOIN patients p ON v.patient_id = p.patient_id 
    GROUP BY p.name HAVING COUNT(*) > 1""", 
) 
  
# ---------- Step 3: Update and delete ---------- 
cur.execute( 
    "UPDATE doctors SET department = 'Internal Medicine' WHERE doctor_id = 3" 
) 
cur.execute("DELETE FROM visits WHERE visit_id = 10") 
conn.commit() 
print("\nUpdated doctor 3 department; deleted visit 10.") 
run("Q5. Doctors table after update", "SELECT * FROM doctors") 
print( 
    "\nTotal visits after delete:", 
    cur.execute("SELECT COUNT(*) FROM visits").fetchone()[0], 
) 
  
# ---------- Step 4: Load into pandas for analysis ---------- 
df = pd.read_sql_query( 
    """ 
    SELECT p.city, p.gender, v.fee FROM visits v 
    JOIN patients p ON v.patient_id = p.patient_id""", 
    conn, 
) 
print("\n--- City-wise average fee using pandas ---") 
print(df.groupby("city")["fee"].agg(["count", "mean"]).round(2).to_string()) 
  
# ---------- Step 5: Same data as a NoSQL document collection ---------- 
import json 
  
rows = pd.read_sql_query( 
    """ 
    SELECT p.patient_id, p.name, p.age, p.city, d.department, v.visit_date, v.fee 
    FROM patients p JOIN visits v ON p.patient_id = v.patient_id 
    JOIN doctors d ON v.doctor_id = d.doctor_id ORDER BY p.patient_id, v.visit_date""", 
    conn, 
) 
conn.close() 
  
collection = [] 
for pid, grp in rows.groupby("patient_id"): 
    first = grp.iloc[0] 
    collection.append( 
        { 
            "_id": int(pid), 
            "name": first["name"], 
            "age": int(first["age"]), 
            "city": first["city"], 
            "visits": [ 
                {"dept": r.department, "date": r.visit_date, "fee": float(r.fee)} 
                for r in grp.itertuples() 
            ], 
        } 
    ) 
with open("patients_nosql.json", "w") as f: 
    json.dump(collection, f, indent=2) 
  
print("\n--- NoSQL document model (one document per patient) ---") 
print(json.dumps(collection[0], indent=2)) 
  
  
def find(coll, condition): 
    """Mimics MongoDB's db.collection.find() using a Python predicate.""" 
    return [doc for doc in coll if condition(doc)] 
  
  
print( 
    "\nMongoDB query : db.patients.find({city: 'Chennai', 'visits.dept': 'Cardiology'})" 
) 
result = find( 
    collection, 
    lambda d: d["city"] == "Chennai" 
    and any(v["dept"] == "Cardiology" for v in d["visits"]), 
) 
for doc in result: 
    print(f"  {doc['_id']}  {doc['name']:<8} visits = {len(doc['visits'])}") 

Hospital Patient Records: Working with a Relational Database

--- Q1. Patients older than 40 ---
SQL: SELECT name, age, city FROM patients WHERE age > 40 ORDER BY age DESC
   name  age    city
Charles   67 Chennai
 Elango   54 Madurai
   Ajay   45 Chennai

--- Q2. Visits with patient and doctor names (JOIN) ---
SQL: SELECT v.visit_id, p.name AS patient, d.name AS doctor, d.department, v.fee FROM visits v JOIN patients p ON v.patient_id = p.patient_id JOIN doctors d ON v.doctor_id = d.doctor_id ORDER BY v.visit_id
 visit_id patient      doctor       department   fee
        1    Ajay  Dr. Ramesh       Cardiology 800.0
        2 Bhavana Dr. Kavitha     Orthopaedics 600.0
        3 Charles  Dr. Ramesh       Cardiology 800.0
        4   Deepa   Dr. Imran General Medicine 300.0
        5  Elango  Dr. Ramesh       Cardiology 800.0
        6    Ajay   Dr. Imran General Medicine 300.0
        7 Fathima Dr. Kavitha     Orthopaedics 600.0
        8 Charles   Dr. Imran General Medicine 300.0
    